# 06 - Benchmark final: konfigurasi terkunci di split test

Alur: set jumlah thread → periksa hash Kunci konfigurasi dan kesamaan env_id dengan run val → periksa penjaga test (K12) → baru buka split test → exact (dan exact ulang), HNSW, IVF, LSH dengan konfigurasi terkunci: bangun, cari, nilai 12 metrik, ukur p50, QPS, dan ukuran index → tambah baris run test → periksa tanda berhasil D4 → catat Lingkungan.

Keluaran:
- `outputs/metrics/runs_test.csv` — 5 baris per eksekusi (exact, exact ulang, HNSW, IVF, LSH), hanya ditambah; setiap baris membawa `config_lock_hash`, `test_attempt`, `reopen_reason` (K4, 007a)
- `outputs/tuning/environment_<env_id>.json` — catatan Lingkungan notebook ini ditambahkan

Prasyarat: `05_val_results` sudah menulis `locked_config.json`; dijalankan di instance yang sama dengan run val (env_id sama, K8 004a).

Ini satu-satunya notebook yang membuka split test, dan test dibuka sekali (K2). Notebook berhenti sebelum membaca query test kalau hash Kunci konfigurasi tidak cocok, env_id berbeda, atau `runs_test.csv` sudah berisi percobaan untuk embedding_id yang sama tanpa izin buka ulang (K12, 007a). Untuk membuka ulang, Arya mengisi `IZIN_BUKA_ULANG = True` dan `ALASAN_BUKA_ULANG` di sel konstanta, lalu mengembalikan izin ke `False` setelah percobaan selesai. Hasil exact pertama menjadi Tetangga exact split test, dan exact dijalankan ulang di notebook yang sama untuk memeriksa nDCG@5 identik (D4).

Keputusan: K2, D4, K10, K11 (003a), K12 dan kolom Run test (007a), K3, K4, K7 (002a, 003a), K6 (002a), K8 dan seed (004a), H4–H6 dan cgroup (005a), pemanasan QPS (006a), K9 (003a).

In [ ]:
from pathlib import Path

# Keluaran 01_preprocessing: queries/qrels.parquet
PROCESSED_DIR = Path("../data/processed")
# Folder induk Set embedding; embedding_id diambil dari Kunci konfigurasi
EMBEDDINGS_DIR = Path("../data/embeddings")
# Folder Kunci konfigurasi dan catatan Lingkungan (K8, K9)
TUNING_DIR = Path("../outputs/tuning")
# Kunci konfigurasi keluaran 05_val_results
LOCKED_CONFIG_PATH = TUNING_DIR / "locked_config.json"
# Folder hasil benchmark final; catatan run test (Kontrak berkas)
METRICS_DIR = Path("../outputs/metrics")
# Root project, dipakai untuk membaca total dan sisa disk
PROJECT_ROOT = Path("..")
# Nama notebook di catatan Lingkungan
NOTEBOOK_NAME = "06_final_benchmark"

# Split yang dibuka notebook ini, sekali (K2)
SPLIT = "test"
# Algoritma yang wajib ada di Kunci konfigurasi dan di hasil test (D4)
ALGORITHMS = ["flat", "hnsw", "ivf", "lsh"]
# Jumlah hasil per query untuk semua algoritma (K7)
K = 5
# Dokumen relevan kalau relevance di qrels paling kecil nilai ini (002a)
MIN_RELEVANCE = 1

# Suku #8 dengan jarak exact sebesar ini atau lebih kecil dikeluarkan (005a, titik periksa 1a)
NEAR_ZERO_DISTANCE = 1e-3
# Jarak untuk slot −1: jarak L2 maksimum dua vektor bernorma 1 (002a)
MISSING_RESULT_DISTANCE = 2.0
# Query pemanasan p50, diambil dari awal split, tidak dihitung (005a)
LATENCY_WARMUP_QUERIES = 10
# Jumlah putaran pengukuran p50 atas semua query split (005a)
LATENCY_ROUNDS = 3
# Jumlah ulangan QPS setelah satu panggilan pemanasan T0 (005a, 006a)
QPS_REPEATS = 5

# Kolom 12 metrik K7 yang wajib terisi untuk setiap algoritma (D4)
METRIC_COLUMNS = [
    "ndcg_at_5",
    "recall_at_5",
    "mrr_at_5",
    "precision_at_5",
    "map_at_5",
    "hit_rate_at_5",
    "knn_recall_at_5",
    "relative_distance_error",
    "qps",
    "latency_p50_ms",
    "build_seconds",
    "index_size_bytes",
]

# Folder induk cgroup: v2 langsung di sini, v1 di subfolder cpu/ dan memory/ (K6, K8, 005a)
CGROUP_DIR = Path("/sys/fs/cgroup")
# Batas RAM cgroup v1 sebesar ini atau lebih dianggap tanpa batas, sama seperti "max" di v2 (005a)
MEMORY_UNLIMITED_FROM = 2**62
# Sumber model CPU dan flag AVX (K8)
CPUINFO_PATH = Path("/proc/cpuinfo")
# Batas atas jumlah thread (K6)
MAX_THREADS = 24
# Paket yang versinya ikut membentuk env_id; daftar ini sama di semua notebook (K8)
ENVIRONMENT_LIBRARIES = [
    "torch",
    "faiss-cpu",
    "sentence-transformers",
    "transformers",
    "datasets",
    "numpy",
    "pandas",
    "pyarrow",
]
# Panjang env_id: awal hash SHA-256 field statis Lingkungan, dalam karakter heksadesimal
ENV_ID_LENGTH = 16

# Catatan run test, juga dibaca penjaga test (K12)
TEST_RUNS_PATH = METRICS_DIR / "runs_test.csv"
# Izin membuka ulang split test untuk embedding_id yang sudah punya percobaan; bawaan mati (K12, 007a)
IZIN_BUKA_ULANG = False
# Alasan tertulis buka ulang, wajib diisi kalau IZIN_BUKA_ULANG aktif; dicatat di setiap baris run test (K12)
ALASAN_BUKA_ULANG = ""

## 1. Thread

Fungsi disalin identik dari notebook sebelumnya (K6, 005a, H18).

In [ ]:
import math
import os

import faiss
import torch


def parse_cpu_max(text: str) -> float | None:
    """Ubah isi cgroup v2 `cpu.max` menjadi jumlah CPU.

    Args:
        text: Isi berkas, contoh "400000 100000" (4 CPU) atau "max 100000" (tanpa batas).

    Returns:
        Kuota dalam satuan CPU, atau None kalau tanpa batas.
    """
    quota, period = text.split()
    return None if quota == "max" else int(quota) / int(period)


def parse_cfs_quota(quota_text: str, period_text: str) -> float | None:
    """Ubah isi cgroup v1 `cpu.cfs_quota_us` dan `cpu.cfs_period_us` menjadi jumlah CPU.

    Args:
        quota_text: Isi cpu.cfs_quota_us, contoh "400000", atau "-1" (tanpa batas).
        period_text: Isi cpu.cfs_period_us, contoh "100000".

    Returns:
        Kuota dalam satuan CPU, atau None kalau tanpa batas.
    """
    quota = int(quota_text.strip())
    return None if quota < 0 else quota / int(period_text.strip())


def fetch_cgroup_cpu_quota(cgroup_dir: Path) -> float | None:
    """Baca kuota CPU dari cgroup v2 atau v1.

    Args:
        cgroup_dir: Folder induk cgroup.

    Returns:
        Kuota dalam satuan CPU, atau None kalau tanpa batas.

    Raises:
        ValueError: Berkas v2 dan v1 sama-sama ada (host hybrid, urutan deteksi belum diputuskan, H18).
        FileNotFoundError: Berkas v2 maupun v1 tidak ada.
    """
    v2_path = cgroup_dir / "cpu.max"
    v1_quota_path = cgroup_dir / "cpu" / "cpu.cfs_quota_us"
    v1_period_path = cgroup_dir / "cpu" / "cpu.cfs_period_us"
    has_v2 = v2_path.is_file()
    has_v1 = v1_quota_path.is_file() and v1_period_path.is_file()
    if has_v2 and has_v1:
        raise ValueError(
            f"{v2_path} dan {v1_quota_path} sama-sama ada; urutan deteksi host hybrid belum diputuskan (H18)"
        )
    if has_v2:
        return parse_cpu_max(v2_path.read_text(encoding="utf-8"))
    if has_v1:
        return parse_cfs_quota(v1_quota_path.read_text(encoding="utf-8"), v1_period_path.read_text(encoding="utf-8"))
    raise FileNotFoundError(f"Tidak ada {v2_path} maupun {v1_quota_path}; butuh cgroup v1 atau v2 di Linux (H12)")


def fetch_cpu_quota(cgroup_dir: Path) -> float:
    """Baca jatah CPU efektif instance.

    Args:
        cgroup_dir: Folder induk cgroup.

    Returns:
        min(kuota cgroup, jumlah CPU sched_getaffinity), atau jumlah CPU affinity kalau kuota tanpa batas.

    Raises:
        OSError: Bukan Linux (os.sched_getaffinity tidak ada).
    """
    if not hasattr(os, "sched_getaffinity"):
        raise OSError("os.sched_getaffinity tidak tersedia; jatah CPU hanya bisa dibaca di Linux (H12)")
    cgroup_quota = fetch_cgroup_cpu_quota(cgroup_dir)
    affinity_count = float(len(os.sched_getaffinity(0)))
    return affinity_count if cgroup_quota is None else min(cgroup_quota, affinity_count)


def compute_thread_count(cpu_quota: float, max_threads: int) -> int:
    """Hitung jumlah thread dari jatah CPU (K6).

    Args:
        cpu_quota: Jatah CPU efektif.
        max_threads: Batas atas thread.

    Returns:
        ⌊min(cpu_quota, max_threads)⌋, paling sedikit 1.
    """
    return max(1, math.floor(min(cpu_quota, max_threads)))


def set_num_threads(count: int) -> None:
    """Set jumlah thread FAISS dan torch.

    Args:
        count: Jumlah thread.
    """
    faiss.omp_set_num_threads(count)
    torch.set_num_threads(count)


cpu_quota = fetch_cpu_quota(CGROUP_DIR)
thread_count = compute_thread_count(cpu_quota, MAX_THREADS)
set_num_threads(thread_count)
print({"cpu_quota": cpu_quota, "thread_count": thread_count})

## 2. Periksa Kunci konfigurasi dan Lingkungan sebelum membuka test

Hash isi Kunci konfigurasi dihitung ulang dengan `compute_lock_hash` (salinan dari `05_val_results`) dan harus sama dengan `content_hash` yang tersimpan. env_id Lingkungan sekarang harus sama dengan env_id run val di Kunci konfigurasi (K2, K8). Kalau salah satunya gagal, notebook berhenti sebelum query test dibaca.

In [ ]:
import hashlib
import importlib.metadata
import json
import platform
import resource
import shutil
import socket
import subprocess
from datetime import datetime, timezone

import pandas as pd


def load_parquet(path: Path) -> pd.DataFrame:
    """Baca satu tabel Parquet.

    Args:
        path: Berkas Parquet.

    Returns:
        Isi tabel.

    Raises:
        FileNotFoundError: Berkas tidak ada.
    """
    if not path.is_file():
        raise FileNotFoundError(f"Berkas tidak ditemukan: {path}")
    return pd.read_parquet(path)


def load_json(path: Path) -> dict[str, object]:
    """Baca satu berkas JSON.

    Args:
        path: Berkas JSON.

    Returns:
        Isi berkas.

    Raises:
        FileNotFoundError: Berkas tidak ada.
    """
    if not path.is_file():
        raise FileNotFoundError(f"Berkas tidak ditemukan: {path}")
    with path.open(encoding="utf-8") as file:
        return json.load(file)


def fetch_embedding_dir(embeddings_dir: Path, embedding_id: str | None) -> Path:
    """Tentukan folder Set embedding yang dipakai.

    Args:
        embeddings_dir: Folder induk Set embedding.
        embedding_id: embedding_id yang diminta, atau None untuk memakai satu-satunya subfolder.

    Returns:
        Folder Set embedding.

    Raises:
        FileNotFoundError: Folder yang diminta tidak ada, atau tidak ada subfolder sama sekali.
        ValueError: embedding_id None tetapi ada lebih dari satu subfolder.
    """
    if embedding_id is not None:
        target = embeddings_dir / embedding_id
        if not (target / "embedding_set.json").is_file():
            raise FileNotFoundError(f"Set embedding tidak ditemukan: {target}")
        return target
    candidates = sorted(path.parent for path in embeddings_dir.glob("*/embedding_set.json"))
    if not candidates:
        raise FileNotFoundError(f"Tidak ada Set embedding di {embeddings_dir}; jalankan 02_embedding dulu")
    if len(candidates) > 1:
        raise ValueError(f"Ada {len(candidates)} Set embedding di {embeddings_dir}; isi EMBEDDING_ID")
    return candidates[0]


def compute_lock_hash(content: dict[str, object]) -> str:
    """Hitung hash SHA-256 isi kanonik Kunci konfigurasi.

    Args:
        content: Isi Kunci konfigurasi; kunci content_hash diabaikan.

    Returns:
        Hash heksadesimal 64 karakter.
    """
    body = {key: value for key, value in content.items() if key != "content_hash"}
    canonical = json.dumps(body, ensure_ascii=False, sort_keys=True, separators=(",", ":"))
    return hashlib.sha256(canonical.encode("utf-8")).hexdigest()


def parse_nvidia_smi(text: str) -> list[dict[str, str | int]]:
    """Ubah keluaran nvidia-smi menjadi daftar GPU.

    Args:
        text: Keluaran `nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader,nounits`,
            satu baris per GPU, contoh "NVIDIA GeForce RTX 3090, 24576, 550.54.14".

    Returns:
        Per GPU: nama, VRAM total (MiB), versi driver.
    """
    gpus = []
    for line in text.strip().splitlines():
        name, memory_total, driver = (part.strip() for part in line.rsplit(",", 2))
        gpus.append({"name": name, "memory_total_mib": int(memory_total), "driver": driver})
    return gpus


def fetch_gpu_info() -> list[dict[str, str | int]]:
    """Baca model GPU, VRAM total, dan driver lewat nvidia-smi.

    Returns:
        Daftar GPU.
    """
    command = ["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader,nounits"]
    return parse_nvidia_smi(subprocess.run(command, capture_output=True, text=True, check=True).stdout)


def parse_cpuinfo(text: str) -> dict[str, str | bool]:
    """Ambil model CPU dan flag AVX dari isi /proc/cpuinfo.

    Args:
        text: Isi /proc/cpuinfo.

    Returns:
        Model CPU, ada tidaknya flag avx2 dan avx512f.

    Raises:
        ValueError: Baris "model name" atau "flags" tidak ada.
    """
    fields = {}
    for line in text.splitlines():
        key, _, value = line.partition(":")
        fields.setdefault(key.strip(), value.strip())
    if "model name" not in fields or "flags" not in fields:
        raise ValueError("/proc/cpuinfo tidak memuat 'model name' atau 'flags'")
    flags = set(fields["flags"].split())
    return {"model": fields["model name"], "avx2": "avx2" in flags, "avx512f": "avx512f" in flags}


def parse_memory_max(text: str) -> int | None:
    """Ubah isi cgroup v2 `memory.max` menjadi byte.

    Args:
        text: Isi berkas, contoh "34359738368" atau "max".

    Returns:
        Jatah RAM dalam byte, atau None kalau tanpa batas.
    """
    value = text.strip()
    return None if value == "max" else int(value)


def parse_memory_limit(text: str, unlimited_from: int) -> int | None:
    """Ubah isi cgroup v1 `memory.limit_in_bytes` menjadi byte.

    Args:
        text: Isi berkas, contoh "34359738368", atau angka sekitar 2^63 untuk tanpa batas.
        unlimited_from: Nilai mulai dari angka ini dianggap tanpa batas.

    Returns:
        Jatah RAM dalam byte, atau None kalau tanpa batas.
    """
    value = int(text.strip())
    return None if value >= unlimited_from else value


def fetch_memory_quota(cgroup_dir: Path, unlimited_from: int) -> int | None:
    """Baca jatah RAM dari cgroup v2 atau v1.

    Args:
        cgroup_dir: Folder induk cgroup.
        unlimited_from: Batas v1 yang dianggap tanpa batas.

    Returns:
        Jatah RAM dalam byte, atau None kalau tanpa batas.

    Raises:
        ValueError: Berkas v2 dan v1 sama-sama ada (host hybrid, urutan deteksi belum diputuskan, H18).
        FileNotFoundError: Berkas v2 maupun v1 tidak ada.
    """
    v2_path = cgroup_dir / "memory.max"
    v1_path = cgroup_dir / "memory" / "memory.limit_in_bytes"
    has_v2, has_v1 = v2_path.is_file(), v1_path.is_file()
    if has_v2 and has_v1:
        raise ValueError(f"{v2_path} dan {v1_path} sama-sama ada; urutan deteksi host hybrid belum diputuskan (H18)")
    if has_v2:
        return parse_memory_max(v2_path.read_text(encoding="utf-8"))
    if has_v1:
        return parse_memory_limit(v1_path.read_text(encoding="utf-8"), unlimited_from)
    raise FileNotFoundError(f"Tidak ada {v2_path} maupun {v1_path}; butuh cgroup v1 atau v2 di Linux (H12)")


def fetch_library_versions(names: list[str]) -> dict[str, str]:
    """Baca versi paket terpasang tanpa meng-import paketnya.

    Args:
        names: Nama distribusi paket.

    Returns:
        Nama paket ke versi.
    """
    return {name: importlib.metadata.version(name) for name in names}


def collect_environment(
    cpu_quota: float, revisions: dict[str, object], project_root: Path, cgroup_dir: Path, cpuinfo_path: Path
) -> dict[str, dict[str, object]]:
    """Kumpulkan field statis dan dinamis Lingkungan (K8, 004a).

    Args:
        cpu_quota: Jatah CPU efektif dari fetch_cpu_quota.
        revisions: Revision model dan dataset.
        project_root: Root project untuk ukuran disk.
        cgroup_dir: Folder cgroup v2.
        cpuinfo_path: Berkas /proc/cpuinfo.

    Returns:
        {"static": field yang membentuk env_id, "dynamic": field yang hanya dicatat}.
    """
    disk = shutil.disk_usage(project_root)
    static = {
        "gpus": fetch_gpu_info(),
        "cuda": torch.version.cuda,
        "cpu": parse_cpuinfo(cpuinfo_path.read_text(encoding="utf-8")),
        "cpu_quota": cpu_quota,
        "cpu_total": os.cpu_count(),
        "memory_quota_bytes": fetch_memory_quota(cgroup_dir, MEMORY_UNLIMITED_FROM),
        "disk_total_bytes": disk.total,
        "threads_faiss": faiss.omp_get_max_threads(),
        "threads_torch": torch.get_num_threads(),
        "python": platform.python_version(),
        "libraries": fetch_library_versions(ENVIRONMENT_LIBRARIES),
        "revisions": revisions,
        "hostname": socket.gethostname(),
    }
    dynamic = {
        # ru_maxrss di Linux dalam KiB
        "peak_ram_bytes": resource.getrusage(resource.RUSAGE_SELF).ru_maxrss * 1024,
        "peak_vram_bytes": torch.cuda.max_memory_allocated() if torch.cuda.is_available() else None,
        "disk_free_bytes": disk.free,
        # Cap waktu UTC format ISO 8601, contoh 2026-10-02T08:30:00+00:00
        "timestamp": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    }
    return {"static": static, "dynamic": dynamic}


def compute_env_id(static: dict[str, object], length: int) -> str:
    """Hitung env_id dari field statis Lingkungan.

    Args:
        static: Field statis, termasuk hostname.
        length: Jumlah karakter heksadesimal yang diambil dari awal hash.

    Returns:
        Awal SHA-256 dari JSON field statis berkunci terurut.
    """
    canonical = json.dumps(static, ensure_ascii=False, sort_keys=True, separators=(",", ":"))
    return hashlib.sha256(canonical.encode("utf-8")).hexdigest()[:length]


def save_environment(environment: dict[str, dict[str, object]], env_id: str, notebook: str, tuning_dir: Path) -> Path:
    """Tambahkan catatan Lingkungan ke `environment_<env_id>.json`.

    Args:
        environment: Hasil collect_environment.
        env_id: Hasil compute_env_id.
        notebook: Nama notebook pencatat.
        tuning_dir: Folder outputs/tuning.

    Returns:
        Path berkas Lingkungan.

    Raises:
        ValueError: Berkas env_id yang sama sudah ada tetapi field statisnya berbeda.
    """
    path = tuning_dir / f"environment_{env_id}.json"
    record = {"notebook": notebook, **environment["dynamic"]}
    if path.is_file():
        with path.open(encoding="utf-8") as file:
            content = json.load(file)
        if content["static"] != json.loads(json.dumps(environment["static"])):
            raise ValueError(f"{path}: field statis berbeda untuk env_id yang sama")
        content["records"].append(record)
    else:
        content = {"env_id": env_id, "static": environment["static"], "records": [record]}
    tuning_dir.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8", newline="\n") as file:
        json.dump(content, file, ensure_ascii=False, indent=2)
    return path


def validate_locked_config(locked_config: dict[str, object], algorithms: list[str]) -> None:
    """Pastikan Kunci konfigurasi utuh dan memuat semua algoritma.

    Args:
        locked_config: Isi locked_config.json.
        algorithms: Algoritma yang wajib ada.

    Raises:
        ValueError: Hash isi tidak cocok, split bukan val, atau ada algoritma yang tidak terkunci.
    """
    if compute_lock_hash(locked_config) != locked_config.get("content_hash"):
        raise ValueError("Hash isi Kunci konfigurasi tidak cocok; split test tidak dibuka")
    if locked_config.get("split") != "val":
        raise ValueError(f"Kunci konfigurasi berasal dari split {locked_config.get('split')!r}, seharusnya 'val'")
    missing = sorted(set(algorithms) - set(locked_config["configs"]))
    if missing:
        raise ValueError(f"Algoritma tanpa konfigurasi terkunci: {missing}")


def validate_environment(current_env_id: str, locked_env_id: str) -> None:
    """Pastikan Lingkungan sekarang sama dengan Lingkungan run val (K8).

    Args:
        current_env_id: env_id instance sekarang.
        locked_env_id: env_id run val di Kunci konfigurasi.

    Raises:
        ValueError: env_id berbeda.
    """
    if current_env_id != locked_env_id:
        raise ValueError(
            f"env_id sekarang {current_env_id} berbeda dengan run val {locked_env_id}; split test tidak dibuka"
        )


locked_config = load_json(LOCKED_CONFIG_PATH)
validate_locked_config(locked_config, ALGORITHMS)
embedding_dir = fetch_embedding_dir(EMBEDDINGS_DIR, locked_config["embedding_id"])
embedding_set = load_json(embedding_dir / "embedding_set.json")
embedding_id = embedding_set["embedding_id"]
embedding_config = embedding_set["config"]
revisions = {"model": embedding_config["model_revision"], "datasets": embedding_config["dataset_revisions"]}
environment = collect_environment(cpu_quota, revisions, PROJECT_ROOT, CGROUP_DIR, CPUINFO_PATH)
env_id = compute_env_id(environment["static"], ENV_ID_LENGTH)
validate_environment(env_id, locked_config["env_id"])
print({"content_hash": locked_config["content_hash"], "env_id": env_id, "embedding_id": embedding_id})

## 3. Penjaga test (K12)

Sebelum query test dibaca, `runs_test.csv` diperiksa untuk embedding_id Kunci konfigurasi, apa pun Kunci-nya (007a titik periksa 2a). Belum ada percobaan → percobaan 1. Sudah ada dan izin mati → berhenti. Izin aktif tanpa alasan → berhenti. Izin aktif dengan alasan → peringatan, percobaan berikutnya, alasan dicatat (007a titik periksa 1a). Tahap ini hanya membaca catatan run test, bukan data test.

In [ ]:
def load_test_runs(path: Path) -> pd.DataFrame:
    """Baca catatan run test yang sudah ada.

    Args:
        path: runs_test.csv.

    Returns:
        Isi berkas, atau tabel kosong berkolom embedding_id, test_attempt, timestamp kalau berkas belum ada.
    """
    if not path.is_file():
        return pd.DataFrame({"embedding_id": [], "test_attempt": [], "timestamp": []})
    return pd.read_csv(path, dtype={"embedding_id": str})


def decide_test_attempt(previous: pd.DataFrame, embedding_id: str, allow_reopen: bool, reason: str) -> int:
    """Tentukan nomor percobaan test, atau hentikan notebook kalau test sudah pernah dibuka (K12).

    Args:
        previous: Catatan run test yang sudah ada.
        embedding_id: embedding_id Kunci konfigurasi.
        allow_reopen: Nilai IZIN_BUKA_ULANG.
        reason: Nilai ALASAN_BUKA_ULANG.

    Returns:
        1 kalau belum ada percobaan untuk embedding_id itu, selain itu nomor percobaan terbesar + 1.

    Raises:
        ValueError: Catatan lama tanpa kolom test_attempt, percobaan sudah ada dan izin mati, atau izin aktif
            tanpa alasan.
    """
    if "test_attempt" not in previous.columns:
        raise ValueError("runs_test.csv tidak punya kolom test_attempt; baris lama tidak diubah (K4)")
    attempts = previous.loc[previous["embedding_id"] == embedding_id]
    if attempts.empty:
        return 1
    last_attempt = int(attempts["test_attempt"].max())
    summary = f"{last_attempt} percobaan, terakhir {attempts['timestamp'].max()}"
    if not allow_reopen:
        raise ValueError(
            f"Split test untuk embedding_id {embedding_id} sudah dibuka ({summary}); "
            "isi IZIN_BUKA_ULANG = True dan ALASAN_BUKA_ULANG untuk membuka ulang"
        )
    if not reason.strip():
        raise ValueError("IZIN_BUKA_ULANG aktif tetapi ALASAN_BUKA_ULANG kosong; split test tidak dibuka")
    print(f"PERINGATAN: membuka ulang split test untuk embedding_id {embedding_id} ({summary}); alasan: {reason}")
    return last_attempt + 1


test_attempt = decide_test_attempt(load_test_runs(TEST_RUNS_PATH), embedding_id, IZIN_BUKA_ULANG, ALASAN_BUKA_ULANG)
reopen_reason = ALASAN_BUKA_ULANG if test_attempt > 1 else ""
print({"embedding_id": embedding_id, "test_attempt": test_attempt, "reopen_reason": reopen_reason})

## 4. Buka split test

Satu-satunya tempat query test dibaca (K2).

In [ ]:
import numpy as np


def load_vectors(path: Path) -> np.ndarray:
    """Baca matriks vektor .npy sebagai memory map baca-saja.

    Args:
        path: Berkas .npy float32.

    Returns:
        Matriks vektor.

    Raises:
        FileNotFoundError: Berkas tidak ada.
    """
    if not path.is_file():
        raise FileNotFoundError(f"Berkas tidak ditemukan: {path}")
    return np.load(path, mmap_mode="r")


def select_split_rows(query_ids: pd.DataFrame, queries: pd.DataFrame, split: str) -> np.ndarray:
    """Cari baris vektor query yang termasuk satu split.

    Args:
        query_ids: Urutan query_id baris vektor query (query_ids.parquet).
        queries: Tabel Query dengan kolom query_id dan split.
        split: Nama split.

    Returns:
        Indeks baris vektor query untuk split itu, menaik.

    Raises:
        ValueError: Ada query_id vektor yang tidak ada di tabel Query.
    """
    split_by_id = queries.set_index("query_id")["split"]
    missing = query_ids.loc[~query_ids["query_id"].isin(split_by_id.index), "query_id"]
    if len(missing):
        raise ValueError(
            f"{len(missing)} query_id vektor tidak ada di queries.parquet, contoh: {missing.head(5).tolist()}"
        )
    splits = split_by_id.loc[query_ids["query_id"]].to_numpy()
    return np.flatnonzero(splits == split)


def build_relevance(qrels: pd.DataFrame, min_relevance: int) -> dict[str, set[str]]:
    """Kelompokkan dokumen relevan per query.

    Args:
        qrels: Tabel Penilaian relevansi.
        min_relevance: Nilai relevance terkecil yang dihitung relevan.

    Returns:
        query_id ke himpunan doc_id relevan.
    """
    relevant = qrels.loc[qrels["relevance"] >= min_relevance]
    return {query_id: set(group["doc_id"]) for query_id, group in relevant.groupby("query_id")}


doc_vectors = load_vectors(embedding_dir / "doc_vectors.npy")
query_vectors = load_vectors(embedding_dir / "query_vectors.npy")
doc_ids = load_parquet(embedding_dir / "doc_ids.parquet")["doc_id"].to_numpy()
query_ids_frame = load_parquet(embedding_dir / "query_ids.parquet")
split_rows = select_split_rows(query_ids_frame, load_parquet(PROCESSED_DIR / "queries.parquet"), SPLIT)
split_query_ids = query_ids_frame["query_id"].to_numpy()[split_rows]
split_vectors = np.ascontiguousarray(query_vectors[split_rows])
relevant = build_relevance(load_parquet(PROCESSED_DIR / "qrels.parquet"), MIN_RELEVANCE)
print({"documents": len(doc_ids), "split": SPLIT, "queries": len(split_query_ids)})

## 5. Penilai dan pengukur

Disalin identik dari `04a_hnsw` dan `04c_lsh` (K7, 005a, 006a). Urutan per konfigurasi sama dengan notebook val: hasil yang dinilai (satu batch, tidak diukur) → p50 → QPS (T₀ lalu T₁..T₅) → #12. LSH memakai `evaluate_lsh_index` (jarak L2 dihitung ulang dari vektor asli).

In [ ]:
def search_index(index: faiss.Index, queries: np.ndarray, k: int) -> tuple[np.ndarray, np.ndarray]:
    """Cari top-k semua query dalam satu panggilan.

    Args:
        index: Index FAISS.
        queries: Vektor query float32.
        k: Jumlah hasil per query.

    Returns:
        Skor (L2² untuk index L2) dan indeks baris dokumen; indeks −1 untuk slot kosong.
    """
    return index.search(queries, k)


def to_doc_ids(indices: np.ndarray, doc_ids: np.ndarray) -> list[list[str | None]]:
    """Ubah indeks baris dokumen menjadi doc_id; slot −1 menjadi None.

    Args:
        indices: Indeks baris dokumen per query.
        doc_ids: doc_id per baris vektor dokumen.

    Returns:
        doc_id per query per peringkat.
    """
    safe = np.where(indices < 0, 0, indices)
    mapped = doc_ids[safe]
    return [[doc_id if index >= 0 else None for doc_id, index in zip(row, raw)] for row, raw in zip(mapped, indices)]


def to_distances(scores: np.ndarray) -> np.ndarray:
    """Ubah skor L2² FAISS menjadi jarak L2; skor negatif kecil dipotong ke 0 (005a).

    Args:
        scores: Skor L2².

    Returns:
        Jarak L2 float64.
    """
    return np.sqrt(np.maximum(scores.astype(np.float64), 0.0))


def to_result_distances(distances: np.ndarray, indices: np.ndarray, missing_distance: float) -> np.ndarray:
    """Beri jarak penalti untuk slot −1.

    Args:
        distances: Jarak L2 hasil pencarian.
        indices: Indeks hasil; −1 menandai slot kosong.
        missing_distance: Jarak untuk slot −1.

    Returns:
        Jarak dengan slot −1 diganti missing_distance.
    """
    return np.where(indices < 0, missing_distance, distances)


def _score_query(retrieved: list[str | None], relevant: set[str], k: int) -> tuple[float, ...]:
    hits = [1.0 if doc_id is not None and doc_id in relevant else 0.0 for doc_id in retrieved[:k]]
    ideal_count = min(len(relevant), k)
    dcg = sum(hit / math.log2(rank + 1) for rank, hit in enumerate(hits, start=1))
    idcg = sum(1.0 / math.log2(rank + 1) for rank in range(1, ideal_count + 1))
    first_hit = next((rank for rank, hit in enumerate(hits, start=1) if hit), None)
    precision_sum = sum(sum(hits[:rank]) / rank for rank, hit in enumerate(hits, start=1) if hit)
    hit_count = sum(hits)
    return (
        dcg / idcg,
        hit_count / len(relevant),
        0.0 if first_hit is None else 1.0 / first_hit,
        hit_count / k,
        precision_sum / ideal_count,
        1.0 if hit_count >= 1 else 0.0,
    )


def compute_quality_metrics(
    retrieved: list[list[str | None]], query_ids: np.ndarray, relevant: dict[str, set[str]], k: int
) -> dict[str, float]:
    """Hitung metrik kualitas #1–#6 rata-rata atas query (K7).

    Args:
        retrieved: doc_id hasil per query, None untuk slot −1.
        query_ids: query_id per baris retrieved.
        relevant: query_id ke himpunan doc_id relevan; setiap query punya paling sedikit satu.
        k: Jumlah hasil yang dinilai.

    Returns:
        nDCG@5, Recall@5, MRR@5, Precision@5, MAP@5, Hit rate@5.
    """
    names = ["ndcg_at_5", "recall_at_5", "mrr_at_5", "precision_at_5", "map_at_5", "hit_rate_at_5"]
    per_query = np.array([_score_query(row, relevant[query_id], k) for row, query_id in zip(retrieved, query_ids)])
    return dict(zip(names, per_query.mean(axis=0).tolist()))


def compute_fidelity_metrics(
    ann_indices: np.ndarray,
    ann_distances: np.ndarray,
    exact_indices: np.ndarray,
    exact_distances: np.ndarray,
    near_zero_distance: float,
) -> dict[str, float | int]:
    """Hitung k-NN Recall@5 (#7) dan Relative distance error (#8) beserta diagnostiknya (K7, 005a).

    Args:
        ann_indices: Indeks hasil algoritma; −1 untuk slot kosong.
        ann_distances: Jarak L2 hasil algoritma, slot −1 sudah diberi penalti.
        exact_indices: Indeks Tetangga exact.
        exact_distances: Jarak L2 Tetangga exact.
        near_zero_distance: Suku dengan jarak exact sebesar ini atau lebih kecil dikeluarkan.

    Returns:
        knn_recall_at_5, relative_distance_error (NaN kalau tidak ada query dengan suku tersisa),
        rde_excluded_terms, rde_queries_without_terms.
    """
    k = exact_indices.shape[1]
    matches = [
        len({index for index in ann_row if index >= 0} & set(exact_row))
        for ann_row, exact_row in zip(ann_indices, exact_indices)
    ]
    sorted_ann = np.sort(ann_distances, axis=1)
    kept = exact_distances > near_zero_distance
    safe_exact = np.where(kept, exact_distances, 1.0)
    errors = np.where(kept, (sorted_ann - exact_distances) / safe_exact, 0.0)
    kept_counts = kept.sum(axis=1)
    has_terms = kept_counts > 0
    per_query = errors[has_terms].sum(axis=1) / kept_counts[has_terms]
    return {
        "knn_recall_at_5": float(np.mean(matches)) / k,
        "relative_distance_error": float(per_query.mean()) if has_terms.any() else float("nan"),
        "rde_excluded_terms": int((k - kept_counts).sum()),
        "rde_queries_without_terms": int((~has_terms).sum()),
    }


def count_short_results(indices: np.ndarray) -> int:
    """Hitung query yang hasilnya kurang dari k (ada slot −1).

    Args:
        indices: Indeks hasil per query.

    Returns:
        Jumlah query dengan paling sedikit satu slot −1.
    """
    return int((indices < 0).any(axis=1).sum())


def measure_latency_p50(index: faiss.Index, queries: np.ndarray, k: int, warmup_count: int, rounds: int) -> float:
    """Ukur latensi p50 pencarian satu query per panggilan (#10, 005a).

    Args:
        index: Index FAISS.
        queries: Vektor query split.
        k: Jumlah hasil per query.
        warmup_count: Jumlah query pemanasan dari awal split, tidak dihitung.
        rounds: Jumlah putaran atas semua query.

    Returns:
        Median waktu index.search per query dalam milidetik.
    """
    single_queries = [queries[row : row + 1] for row in range(len(queries))]
    for query in single_queries[:warmup_count]:
        index.search(query, k)
    timings = []
    for _ in range(rounds):
        for query in single_queries:
            start = time.perf_counter_ns()
            index.search(query, k)
            timings.append(time.perf_counter_ns() - start)
    return float(np.median(timings)) / 1e6


def measure_qps(index: faiss.Index, queries: np.ndarray, k: int, repeats: int) -> float:
    """Ukur QPS pencarian batch semua query (#9, 005a, 006a).

    Args:
        index: Index FAISS, dengan jumlah thread sudah diset.
        queries: Vektor query split.
        k: Jumlah hasil per query.
        repeats: Jumlah ulangan yang diukur setelah pemanasan T0.

    Returns:
        Jumlah query dibagi median waktu ulangan, dalam query per detik.
    """
    index.search(queries, k)
    timings = []
    for _ in range(repeats):
        start = time.perf_counter_ns()
        index.search(queries, k)
        timings.append(time.perf_counter_ns() - start)
    return len(queries) / (float(np.median(timings)) / 1e9)


def measure_index_size(index: faiss.Index) -> int:
    """Ukur ukuran index sebagai jumlah byte hasil serialisasi (#12, 005a).

    Args:
        index: Index FAISS.

    Returns:
        Jumlah byte faiss.serialize_index(index).
    """
    buffer = faiss.serialize_index(index)
    size = len(buffer)
    del buffer
    return size


def evaluate_index(
    index: faiss.Index,
    queries: np.ndarray,
    query_ids: np.ndarray,
    exact_indices: np.ndarray,
    exact_distances: np.ndarray,
    build_seconds: float,
) -> tuple[dict[str, float | int], dict[str, int]]:
    """Nilai dan ukur satu konfigurasi index pada query split.

    Args:
        index: Index FAISS dengan parameter search konfigurasi ini sudah diset.
        queries: Vektor query split.
        query_ids: query_id per baris queries.
        exact_indices: Indeks Tetangga exact.
        exact_distances: Jarak L2 Tetangga exact.
        build_seconds: Waktu build index (#11).

    Returns:
        12 metrik K7 dalam urutan kolom catatan run, dan kolom diagnostik.
    """
    scores, indices = search_index(index, queries, K)
    distances = to_result_distances(to_distances(scores), indices, MISSING_RESULT_DISTANCE)
    quality = compute_quality_metrics(to_doc_ids(indices, doc_ids), query_ids, relevant, K)
    fidelity = compute_fidelity_metrics(indices, distances, exact_indices, exact_distances, NEAR_ZERO_DISTANCE)
    latency_p50_ms = measure_latency_p50(index, queries, K, LATENCY_WARMUP_QUERIES, LATENCY_ROUNDS)
    qps = measure_qps(index, queries, K, QPS_REPEATS)
    metrics = {
        **quality,
        "knn_recall_at_5": fidelity["knn_recall_at_5"],
        "relative_distance_error": fidelity["relative_distance_error"],
        "qps": qps,
        "latency_p50_ms": latency_p50_ms,
        "build_seconds": build_seconds,
        "index_size_bytes": measure_index_size(index),
    }
    diagnostics = {
        "short_result_queries": count_short_results(indices),
        "rde_excluded_terms": fidelity["rde_excluded_terms"],
        "rde_queries_without_terms": fidelity["rde_queries_without_terms"],
    }
    return metrics, diagnostics


def compute_l2_distances(queries: np.ndarray, doc_vectors: np.ndarray, indices: np.ndarray) -> np.ndarray:
    """Hitung ulang jarak L2 antara query dan dokumen hasil dari vektor asli (003a titik periksa 3a).

    Args:
        queries: Vektor query, satu baris per baris indices.
        doc_vectors: Vektor dokumen asli.
        indices: Indeks dokumen hasil; −1 untuk slot kosong.

    Returns:
        Jarak L2 float64 berukuran sama dengan indices; slot −1 bernilai NaN.
    """
    safe = np.where(indices < 0, 0, indices)
    rows = np.asarray(doc_vectors[safe.ravel()], dtype=np.float64).reshape(*indices.shape, -1)
    distances = np.linalg.norm(rows - queries[:, None, :].astype(np.float64), axis=2)
    return np.where(indices < 0, np.nan, distances)


def evaluate_lsh_index(
    index: faiss.Index,
    queries: np.ndarray,
    query_ids: np.ndarray,
    exact_indices: np.ndarray,
    exact_distances: np.ndarray,
    build_seconds: float,
) -> tuple[dict[str, float | int], dict[str, int]]:
    """Nilai dan ukur satu build LSH pada query split; #8 memakai jarak L2 yang dihitung ulang.

    Args:
        index: IndexLSH.
        queries: Vektor query split.
        query_ids: query_id per baris queries.
        exact_indices: Indeks Tetangga exact.
        exact_distances: Jarak L2 Tetangga exact.
        build_seconds: Waktu build index (#11).

    Returns:
        12 metrik K7 dalam urutan kolom catatan run, dan kolom diagnostik.
    """
    _, indices = search_index(index, queries, K)
    l2_distances = compute_l2_distances(queries, doc_vectors, indices)
    distances = to_result_distances(l2_distances, indices, MISSING_RESULT_DISTANCE)
    quality = compute_quality_metrics(to_doc_ids(indices, doc_ids), query_ids, relevant, K)
    fidelity = compute_fidelity_metrics(indices, distances, exact_indices, exact_distances, NEAR_ZERO_DISTANCE)
    latency_p50_ms = measure_latency_p50(index, queries, K, LATENCY_WARMUP_QUERIES, LATENCY_ROUNDS)
    qps = measure_qps(index, queries, K, QPS_REPEATS)
    metrics = {
        **quality,
        "knn_recall_at_5": fidelity["knn_recall_at_5"],
        "relative_distance_error": fidelity["relative_distance_error"],
        "qps": qps,
        "latency_p50_ms": latency_p50_ms,
        "build_seconds": build_seconds,
        "index_size_bytes": measure_index_size(index),
    }
    diagnostics = {
        "short_result_queries": count_short_results(indices),
        "rde_excluded_terms": fidelity["rde_excluded_terms"],
        "rde_queries_without_terms": fidelity["rde_queries_without_terms"],
    }
    return metrics, diagnostics

## 6. Pembangun index

Disalin identik dari `03_exact`, `04a_hnsw`, `04b_ivf`, dan `04c_lsh`. Parameter build diambil dari Kunci konfigurasi; parameter pelatihan IVF yang terbaca dari index harus sama dengan yang terkunci.

In [ ]:
import time


def build_flat_index(vectors: np.ndarray) -> tuple[faiss.IndexFlatL2, float]:
    """Bangun IndexFlatL2 dan ukur waktunya.

    Args:
        vectors: Vektor dokumen float32 bernorma 1.

    Returns:
        Index berisi semua vektor dan waktu build dalam detik (#11).
    """
    start = time.perf_counter_ns()
    index = faiss.IndexFlatL2(vectors.shape[1])
    index.add(vectors)
    return index, (time.perf_counter_ns() - start) / 1e9


def build_hnsw_index(vectors: np.ndarray, m: int, ef_construction: int) -> tuple[faiss.IndexHNSWFlat, float]:
    """Bangun IndexHNSWFlat (METRIC_L2) dan ukur waktunya.

    Args:
        vectors: Vektor dokumen float32 bernorma 1.
        m: Jumlah tetangga per node graf.
        ef_construction: Lebar pencarian saat membangun graf.

    Returns:
        Index berisi semua vektor dan waktu build dalam detik (#11).
    """
    start = time.perf_counter_ns()
    index = faiss.IndexHNSWFlat(vectors.shape[1], m, faiss.METRIC_L2)
    index.hnsw.efConstruction = ef_construction
    index.add(vectors)
    return index, (time.perf_counter_ns() - start) / 1e9


def build_ivf_index(vectors: np.ndarray, nlist: int) -> tuple[faiss.IndexIVFFlat, float, dict[str, int]]:
    """Latih dan bangun IndexIVFFlat (quantizer IndexFlatL2, METRIC_L2) dan ukur waktunya.

    Args:
        vectors: Vektor dokumen float32 bernorma 1.
        nlist: Jumlah sel IVF.

    Returns:
        Index berisi semua vektor, waktu build dalam detik (#11), dan parameter pelatihan:
        seed k-means, max_points_per_centroid, dan jumlah vektor latih hasil sampel FAISS.

    Raises:
        AttributeError: Index tidak punya parameter clustering cp.seed atau cp.max_points_per_centroid.
    """
    start = time.perf_counter_ns()
    quantizer = faiss.IndexFlatL2(vectors.shape[1])
    index = faiss.IndexIVFFlat(quantizer, vectors.shape[1], nlist, faiss.METRIC_L2)
    clustering = getattr(index, "cp", None)
    if not (hasattr(clustering, "seed") and hasattr(clustering, "max_points_per_centroid")):
        raise AttributeError("IndexIVFFlat tidak punya cp.seed / cp.max_points_per_centroid; seed tidak terbaca (004a)")
    index.train(vectors)
    index.add(vectors)
    seconds = (time.perf_counter_ns() - start) / 1e9
    max_points = int(clustering.max_points_per_centroid)
    train_info = {
        "kmeans_seed": int(clustering.seed),
        "max_points_per_centroid": max_points,
        "train_points": min(len(vectors), nlist * max_points),
    }
    return index, seconds, train_info


def build_lsh_index(
    vectors: np.ndarray, nbits: int, rotate_data: bool, train_thresholds: bool
) -> tuple[faiss.IndexLSH, float]:
    """Bangun IndexLSH dan ukur waktunya.

    Args:
        vectors: Vektor dokumen float32 bernorma 1.
        nbits: Panjang kode biner.
        rotate_data: Proyeksi acak sebelum dibinerkan.
        train_thresholds: Latih ambang biner.

    Returns:
        Index berisi semua vektor dan waktu build dalam detik (#11).
    """
    start = time.perf_counter_ns()
    index = faiss.IndexLSH(vectors.shape[1], nbits, rotate_data, train_thresholds)
    if not index.is_trained:
        index.train(vectors)
    index.add(vectors)
    return index, (time.perf_counter_ns() - start) / 1e9


def validate_build_params(algorithm: str, actual: dict[str, object], locked: dict[str, object]) -> None:
    """Pastikan parameter build test sama dengan yang terkunci dari val.

    Args:
        algorithm: Nama algoritma untuk pesan error.
        actual: Parameter build index test.
        locked: Parameter build di Kunci konfigurasi.

    Raises:
        ValueError: Parameter berbeda.
    """
    if actual != locked:
        raise ValueError(f"{algorithm}: parameter build test {actual} berbeda dengan Kunci konfigurasi {locked}")

## 7. Exact dan exact ulang

Exact pertama memberi Tetangga exact split test (dari satu panggilan batch, seperti di `03_exact`) dan satu run. Index exact lalu dibangun ulang dan dinilai lagi sebagai run kedua untuk pemeriksaan D4 (nDCG@5 identik). Index dilepas sebelum index berikutnya dibangun.

In [ ]:
results = []
for attempt in range(2):
    flat_index, build_seconds = build_flat_index(doc_vectors)
    if attempt == 0:
        exact_scores, exact_indices = search_index(flat_index, split_vectors, K)
        exact_distances = to_distances(exact_scores)
    metrics, diagnostics = evaluate_index(flat_index, split_vectors, split_query_ids, exact_indices, exact_distances,
                                          build_seconds)
    del flat_index
    results.append(("flat", {"build": {}, "search": {}}, metrics, diagnostics))
    print({"algorithm": "flat", "attempt": attempt + 1, **metrics, **diagnostics})

## 8. HNSW, IVF, LSH dengan konfigurasi terkunci

Satu build dan satu run per algoritma; parameter search diset sebelum pencarian dan pemanasan T₀ (006a).

In [ ]:
configs = locked_config["configs"]

hnsw_build = configs["hnsw"]["params"]["build"]
hnsw_index, build_seconds = build_hnsw_index(doc_vectors, hnsw_build["M"], hnsw_build["efConstruction"])
hnsw_index.hnsw.efSearch = configs["hnsw"]["params"]["search"]["efSearch"]
metrics, diagnostics = evaluate_index(hnsw_index, split_vectors, split_query_ids, exact_indices, exact_distances,
                                      build_seconds)
del hnsw_index
results.append(("hnsw", configs["hnsw"]["params"], metrics, diagnostics))

ivf_build = configs["ivf"]["params"]["build"]
ivf_index, build_seconds, train_info = build_ivf_index(doc_vectors, ivf_build["nlist"])
validate_build_params("ivf", {"nlist": ivf_build["nlist"], **train_info}, ivf_build)
ivf_index.nprobe = configs["ivf"]["params"]["search"]["nprobe"]
metrics, diagnostics = evaluate_index(ivf_index, split_vectors, split_query_ids, exact_indices, exact_distances,
                                      build_seconds)
del ivf_index
results.append(("ivf", configs["ivf"]["params"], metrics, diagnostics))

lsh_build = configs["lsh"]["params"]["build"]
lsh_index, build_seconds = build_lsh_index(
    doc_vectors, lsh_build["nbits"], lsh_build["rotate_data"], lsh_build["train_thresholds"]
)
metrics, diagnostics = evaluate_lsh_index(lsh_index, split_vectors, split_query_ids, exact_indices, exact_distances,
                                          build_seconds)
del lsh_index
results.append(("lsh", configs["lsh"]["params"], metrics, diagnostics))

for algorithm, params, metrics, diagnostics in results[2:]:
    print({"algorithm": algorithm, **metrics, **diagnostics})

## 9. Catat run test dan periksa tanda berhasil

Setiap baris run ditambah `config_lock_hash` (content_hash Kunci), `test_attempt`, dan `reopen_reason` (K4, 007a). Semua run ditulis dulu ke `outputs/metrics/runs_test.csv` (hanya ditambah), baru tanda berhasil D4 diperiksa pada percobaan ini, yaitu percobaan terakhir (007a), supaya angka tetap tersimpan walaupun pemeriksaan gagal. D4: (1) 12 metrik terisi untuk keempat algoritma; (2) exact: k-NN Recall@5 = 1 dan Relative distance error = 0; (3) nDCG@5 exact ulang identik dengan exact pertama.

In [ ]:
import uuid


def build_run(
    algorithm: str,
    params: dict[str, dict[str, object]],
    metrics: dict[str, float | int],
    diagnostics: dict[str, int],
    identity: dict[str, str],
) -> dict[str, object]:
    """Susun satu baris catatan run (K4).

    Args:
        algorithm: flat, hnsw, ivf, atau lsh.
        params: {"build": ..., "search": ...}; IVF juga jumlah sampel latih dan seed k-means, LSH seed rotasi.
        metrics: 12 metrik K7.
        diagnostics: Kolom diagnostik (hasil < 5, suku #8 dikeluarkan, query tanpa suku #8).
        identity: embedding_id, env_id, split.

    Returns:
        Baris run dengan run_id baru dan cap waktu.
    """
    return {
        "run_id": uuid.uuid4().hex,
        # Cap waktu UTC format ISO 8601, contoh 2026-10-02T08:30:00+00:00
        "timestamp": datetime.now(timezone.utc).isoformat(timespec="seconds"),
        "embedding_id": identity["embedding_id"],
        "env_id": identity["env_id"],
        "algorithm": algorithm,
        "params": json.dumps(params, ensure_ascii=False, sort_keys=True),
        "split": identity["split"],
        "k": K,
        "threads_faiss": faiss.omp_get_max_threads(),
        "threads_torch": torch.get_num_threads(),
        **metrics,
        **diagnostics,
    }


def append_run(run: dict[str, object], path: Path) -> None:
    """Tambahkan satu baris run ke CSV tanpa menimpa baris lama.

    Args:
        run: Baris run.
        path: Berkas CSV catatan run.

    Raises:
        ValueError: Kolom berkas lama berbeda dengan kolom run.
    """
    row = pd.DataFrame([run])
    if path.is_file():
        existing_columns = pd.read_csv(path, nrows=0).columns.tolist()
        if existing_columns != row.columns.tolist():
            raise ValueError(f"Kolom {path} berbeda dengan kolom run baru; baris tidak ditambahkan")
        row.to_csv(path, mode="a", header=False, index=False, lineterminator="\n")
        return
    path.parent.mkdir(parents=True, exist_ok=True)
    row.to_csv(path, index=False, lineterminator="\n")


def validate_success_criteria(runs: list[dict[str, object]], algorithms: list[str], metric_columns: list[str]) -> None:
    """Periksa tanda berhasil benchmark D4.

    Args:
        runs: Baris run test, dua yang pertama exact dan exact ulang.
        algorithms: Algoritma yang wajib ada.
        metric_columns: Kolom 12 metrik.

    Raises:
        ValueError: Salah satu dari tiga syarat D4 tidak terpenuhi; pesan menyebut semua yang gagal.
    """
    failures = []
    if sorted({run["algorithm"] for run in runs}) != sorted(algorithms):
        failures.append(f"algoritma {sorted({run['algorithm'] for run in runs})}")
    for run in runs:
        empty = [column for column in metric_columns if pd.isna(run[column])]
        if empty:
            failures.append(f"{run['algorithm']} {run['run_id']}: metrik kosong {empty}")
    exact_runs = [run for run in runs if run["algorithm"] == "flat"]
    for run in exact_runs:
        if run["knn_recall_at_5"] != 1.0 or run["relative_distance_error"] != 0.0:
            failures.append(f"exact {run['run_id']}: #7 {run['knn_recall_at_5']}, #8 {run['relative_distance_error']}")
    if len(exact_runs) != 2 or exact_runs[0]["ndcg_at_5"] != exact_runs[1]["ndcg_at_5"]:
        failures.append(f"nDCG@5 exact ulang tidak identik: {[run['ndcg_at_5'] for run in exact_runs]}")
    if failures:
        raise ValueError("D4 tidak terpenuhi: " + "; ".join(failures))


def to_test_run(run: dict[str, object], lock_hash: str, attempt: int, reason: str) -> dict[str, object]:
    """Tambahkan kolom khusus run test ke baris run (K4, 007a).

    Args:
        run: Baris run dari build_run.
        lock_hash: content_hash Kunci konfigurasi.
        attempt: Nomor percobaan test.
        reason: Alasan buka ulang; kosong untuk percobaan 1.

    Returns:
        Baris run dengan config_lock_hash, test_attempt, reopen_reason di akhir.
    """
    return {**run, "config_lock_hash": lock_hash, "test_attempt": attempt, "reopen_reason": reason}


identity = {"embedding_id": embedding_id, "env_id": env_id, "split": SPLIT}
test_runs = []
for algorithm, params, metrics, diagnostics in results:
    run = build_run(algorithm, params, metrics, diagnostics, identity)
    run = to_test_run(run, locked_config["content_hash"], test_attempt, reopen_reason)
    append_run(run, TEST_RUNS_PATH)
    test_runs.append(run)
print(pd.DataFrame(test_runs)[["algorithm", "run_id", *METRIC_COLUMNS]].to_string(index=False))
print({"test_attempt": test_attempt, "jumlah_percobaan": test_attempt, "reopen_reason": reopen_reason})
validate_success_criteria(test_runs, ALGORITHMS, METRIC_COLUMNS)
print("D4 terpenuhi pada percobaan", test_attempt)

## 10. Lingkungan

Field dinamis (puncak RAM dan VRAM, sisa disk, cap waktu) dibaca ulang setelah semua pengukuran lalu ditambahkan ke berkas Lingkungan env_id yang sama.

In [ ]:
environment = collect_environment(cpu_quota, revisions, PROJECT_ROOT, CGROUP_DIR, CPUINFO_PATH)
environment_path = save_environment(environment, compute_env_id(environment["static"], ENV_ID_LENGTH), NOTEBOOK_NAME,
                                    TUNING_DIR)
print({"env_id": env_id, "path": str(environment_path), **environment["dynamic"]})